In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = project_root / "data" / "05_bond_book_yield_curve_twist"
if not data_dir.is_dir():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks directory.")

positions = pd.read_csv(data_dir / "positions.csv")
curve = pd.read_csv(data_dir / "curve.csv")
cashflows = pd.read_csv(data_dir / "cashflows.csv")

print(positions)
print(curve)
print(cashflows)


  bond_id  signed_units
0     B03           -50
1     B01            80
2     B02          -120
   maturity_years  zero_rate_pct  shock_bps
0             0.5           2.30         30
1             1.0           2.50         20
2             2.0           2.75        -10
3             3.0           3.00        -35
  bond_id  payment_years  cashflow_eur
0     B01            1.0         100.0
1     B02            0.5           2.0
2     B02            1.0           2.0
3     B02            1.5         102.0
4     B03            0.5           2.5
5     B03            1.0           2.5
6     B03            1.5           2.5
7     B03            2.0           2.5
8     B03            2.5         102.5


In [8]:



def bond_prices(coupons, zero_rates, payment_years_sorted):
    P = np.sum(coupons.iloc[i] * np.exp(-zero_rates.iloc[i] * payment_years_sorted.iloc[i]) for i in range(len(payment_years_sorted)))
    return P

def zero_rates_inter(lower_time, lower_rate, upper_time, upper_rate, middle_time):
    rate = (np.abs(middle_time - lower_time) / (upper_time - lower_time)) * lower_rate + (np.abs(upper_time - middle_time) / (upper_time - lower_time)) * upper_rate
    return rate

print(zero_rates_inter(1, 2.5, 3, 3, 2))

2.75


In [76]:
curve_rename = curve.rename(columns={"maturity_years": "payment_years"})
cashflows_new = cashflows.merge(curve_rename,
                               how="left",
                               on="payment_years")


bond_payment_years_not_in_curve = cashflows_new[cashflows_new["zero_rate_pct"].isna()].reset_index(drop=True)
missing_years = bond_payment_years_not_in_curve["payment_years"].drop_duplicates()


distances = [
    curve["maturity_years"].sub(year).abs() for year in missing_years
]

below = missing_years.iloc[0] > curve["maturity_years"]
above = missing_years.iloc[0] < curve["maturity_years"]

#print(curve.loc[distance[0][above].idxmin()])
for year, distances_for_year in zip(missing_years, distances):
    above = year < curve["maturity_years"]
    below = year > curve["maturity_years"]

    curve.loc[distances_for_year[above].idxmin()]

    #print(year, distances_for_year[above].to_list())
    #print(year, distances_for_year[below].to_list())
    print(curve.loc[distances_for_year[above].idxmin()])


print(curve)
print(missing_years)
print(distances)
print(distances[0][above])
print(distances[0][above].idxmin())



maturity_years     2.00
zero_rate_pct      2.75
shock_bps        -10.00
Name: 2, dtype: float64
maturity_years     3.0
zero_rate_pct      3.0
shock_bps        -35.0
Name: 3, dtype: float64
   maturity_years  zero_rate_pct  shock_bps
0             0.5           2.30         30
1             1.0           2.50         20
2             2.0           2.75        -10
3             3.0           3.00        -35
0    1.5
2    2.5
Name: payment_years, dtype: float64
[0    1.0
1    0.5
2    0.5
3    1.5
Name: maturity_years, dtype: float64, 0    2.0
1    1.5
2    0.5
3    0.5
Name: maturity_years, dtype: float64]
3    1.5
Name: maturity_years, dtype: float64
3


In [20]:
payment_years_sorted = cashflows["payment_years"].sort_values(ascending=True).drop_duplicates()

print(len(payment_years_sorted))

5


In [ ]:
# Creates an empty dictionary to store results
results = {}

# Writes a mask for each `bond_id`-word and stores it in the `results` dictionary
for i in range(len(cashflows["bond_id"].drop_duplicates().reset_index(drop=True))):
    new_results = cashflows["bond_id"] == cashflows["bond_id"].drop_duplicates().reset_index(drop=True).iloc[i]
    results[cashflows["bond_id"].drop_duplicates().reset_index(drop=True).iloc[i]] = new_results
    

print(cashflows.loc[results["B03"]])

  bond_id  payment_years  cashflow_eur
4     B03            0.5           2.5
5     B03            1.0           2.5
6     B03            1.5           2.5
7     B03            2.0           2.5
8     B03            2.5         102.5
